In [10]:
import pandas as pd
import numpy as np

# 1. Cargar datos corrigiendo codificación y separador
df = pd.read_csv('../Data/05_PAS-MOD_TRANSFERENCIA_PNSU.csv', encoding='latin1', sep=';')

# Limpiar columnas no deseadas (Unnamed)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Normalizar nombre de columna de monto (por si tiene espacios extra al final)
df.columns = df.columns.str.strip()



In [11]:
df.head()

,N,CODIGO_PAIS,CODIGO_ENTIDAD,PROGRAMA,CODIGO_SNIP,CODIGO_UNIFICADO,NOMBRE,UNIDAD_EJECUTORA,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_BENEFICIADA_SSP,MONTO_ACTUALIZADO _PIP,MODALIDAD_FINANCIAMIENTO,TIPO_EJECUCION,ETAPA_INVERSION,ESTADO
0,1,PE,11476,PNSU,63002,2056645,"AMPLIACION, MEJORAMIENTO DEL SISTEMA DE AGUA P...",MUNICIPALIDAD DISTRITAL DE ARAMANGO,AMAZONAS,BAGUA,ARAMANGO,2028,"1,644,132.00",Transferencia,Indirecta,OBRA,Paralizada
1,2,PE,11477,PNSU,61273,2066026,MEJORAMIENTO AMPLIACION DEL SISTEMA DE AGUA PO...,MUNICIPALIDAD DISTRITAL DE COPALLIN,AMAZONAS,BAGUA,COPALLIN,6705,"3,071,994.57",Transferencia,Indirecta,OBRA,Concluido
2,3,PE,11478,PNSU,99900,2106401,MEJORAMIENTO Y AMPLIACIÓN DEL SISTEMA DE AGUA ...,REGION AMAZONAS-SEDE CENTRAL,AMAZONAS,BAGUA,IMAZA,2604,"9,562,421.17",Transferencia,Indirecta,OBRA,Concluido
3,4,PE,11479,PNSU,63047,2066683,"MEJORAMIENTO, AMPLIACION DEL SISTEMA DE AGUA P...",MUNICIPALIDAD PROVINCIAL DE BAGUA,AMAZONAS,BAGUA,LA PECA,2050,"4,291,380.74",Transferencia,Indirecta,OBRA,Concluido
4,5,PE,11480,PNSU,2993,2090141,PROYECTO INTEGRADO DEL SISTEMA DE AGUA POTABLE...,GOBIERNO REGIONAL AMAZONAS,AMAZONAS,BAGUA,LA PECA,22556,"219,831,721.64",Transferencia,Indirecta,OBRA (SALDO),Actos Previos


In [8]:
df.dtypes

N                              int64
CODIGO_PAIS                   object
CODIGO_ENTIDAD                 int64
PROGRAMA                      object
CODIGO_SNIP                    int64
CODIGO_UNIFICADO               int64
NOMBRE                        object
UNIDAD_EJECUTORA              object
DEPARTAMENTO                  object
PROVINCIA                     object
DISTRITO                      object
POBLACION_BENEFICIADA_SSP     object
MONTO_ACTUALIZADO _PIP        object
MODALIDAD_FINANCIAMIENTO      object
TIPO_EJECUCION                object
ETAPA_INVERSION               object
ESTADO                        object
Unnamed: 17                  float64
Unnamed: 18                  float64
Unnamed: 19                  float64
Unnamed: 20                  float64
Unnamed: 21                  float64
Unnamed: 22                  float64
Unnamed: 23                  float64
Unnamed: 24                  float64
Unnamed: 25                  float64
dtype: object

In [12]:
# 2. Convertir columnas numéricas
# Reemplazar comas por puntos si existen y convertir a tipo numérico
for col in ['MONTO_ACTUALIZADO _PIP', 'POBLACION_BENEFICIADA_SSP']:
    df[col] = df[col].astype(str).str.replace(',', '.').str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [15]:
# Agrupar por ESTADO de la inversión
resumen_estado = df.groupby('ESTADO').agg(
    total_proyectos=('N', 'count'),
    monto_total_soles=('MONTO_ACTUALIZADO _PIP', 'sum'),
    poblacion_afectada=('POBLACION_BENEFICIADA_SSP', 'sum')
).reset_index()

# Calcular porcentaje del presupuesto total
monto_general = df['MONTO_ACTUALIZADO _PIP'].sum()
resumen_estado['porcentaje_presupuesto'] = (resumen_estado['monto_total_soles'] / monto_general) * 100

print("--- IMPACTO ECONÓMICO Y SOCIAL POR ESTADO DE PROYECTO ---")
print(resumen_estado.sort_values(by='monto_total_soles', ascending=False).to_string(index=False))

--- IMPACTO ECONÓMICO Y SOCIAL POR ESTADO DE PROYECTO ---
           ESTADO  total_proyectos  monto_total_soles  poblacion_afectada  porcentaje_presupuesto
        Concluido             1768       1.305823e+10        18441577.389               61.589920
Convenio Resuelto               34       2.377881e+09         1463842.000               11.215415
       Paralizada               63       2.110796e+09         1255573.000                9.955695
     En Ejecución               39       1.662273e+09          957709.000                7.840208
   En elaboración               20       1.035764e+09          193494.000                4.885244
Contrato Resuelto               13       7.698039e+08          414875.000                3.630826
          Cerrado                2       1.273661e+08           14377.000                0.600730
    Actos Previos                7       5.978150e+07           48586.000                0.281963


In [17]:
# Filtrar solo obras paralizadas
paralizadas = df[df['ESTADO'].str.contains('Paralizada', case=False, na=False)]

# Agrupar por Departamento
ranking_regiones = paralizadas.groupby('DEPARTAMENTO').agg(
    obras_paralizadas=('N', 'count'),
    presupuesto_congelado=('MONTO_ACTUALIZADO _PIP', 'sum'),
    poblacion_sin_servicio=('POBLACION_BENEFICIADA_SSP', 'sum')
).sort_values(by='presupuesto_congelado', ascending=False).reset_index()

print("--- TOP DEPARTAMENTOS CON MAYOR PRESUPUESTO EN OBRAS PARALIZADAS ---")
print(ranking_regiones.head(10).to_string(index=False))

--- TOP DEPARTAMENTOS CON MAYOR PRESUPUESTO EN OBRAS PARALIZADAS ---
DEPARTAMENTO  obras_paralizadas  presupuesto_congelado  poblacion_sin_servicio
  SAN MARTIN                  2           469539751.18                 33763.0
        PUNO                  5           264711814.99                178813.0
       PIURA                  6           229739621.62                170118.0
        LIMA                  5           208017706.72                142408.0
  LAMBAYEQUE                  4           146673139.52                 70336.0
   CAJAMARCA                  2           113027486.43                 27251.0
       JUNIN                  3           108245540.35                 37164.0
    AREQUIPA                  2           103686015.98                 23519.0
     HUANUCO                  5            72933988.65                 29938.0
    AYACUCHO                  2            71850624.57                  7526.0


In [20]:
# (Asumiendo que df ya tiene los montos y población como numéricos)

# ==============================================================================
# PREGUNTA 1: Impacto total del "Fracaso Contractual" (Paralizadas + Resueltas)
# ==============================================================================
estados_fracaso = ['Paralizada', 'Convenio Resuelto', 'Contrato Resuelto']
df_fracaso = df[df['ESTADO'].isin(estados_fracaso)]

monto_bloqueado = df_fracaso['MONTO_ACTUALIZADO _PIP'].sum()
poblacion_afectada_total = df_fracaso['POBLACION_BENEFICIADA_SSP'].sum()
proyectos_afectados = len(df_fracaso)

print("=== 1. RESUMEN DE PROYECTOS EN ENTRAMPAMIENTO (PARALIZADAS Y RESUELTAS) ===")
print(f"Número de proyectos estancados: {proyectos_afectados}")
print(f"Monto total atrapado: S/ {monto_bloqueado:,.2f}")
print(f"Población total sin servicio: {poblacion_afectada_total:,.0f} habitantes\n")

# ==============================================================================
# PREGUNTA 2: Costo por Beneficiario (S/ por persona) por ESTADO
# ==============================================================================
eficiencia = df.groupby('ESTADO').agg(
    monto_total=('MONTO_ACTUALIZADO _PIP', 'sum'),
    poblacion_total=('POBLACION_BENEFICIADA_SSP', 'sum')
).reset_index()

# Calcular costo por persona beneficiada
eficiencia['costo_por_beneficiario'] = eficiencia['monto_total'] / eficiencia['poblacion_total']

print("=== 2. COSTO PROMEDIO DE INVERSIÓN POR PERSONA BENEFICIADA ===")
print(eficiencia[['ESTADO', 'costo_por_beneficiario']].sort_values(by='costo_por_beneficiario', ascending=False).to_string(index=False))
print("\n")

# ==============================================================================
# PREGUNTA 3: Las 10 Unidades Ejecutoras (Alcaldías/GORES) con más presupuesto paralizado
# ==============================================================================
ue_paralizadas = df_fracaso.groupby(['UNIDAD_EJECUTORA', 'DEPARTAMENTO']).agg(
    proyectos_parados=('N', 'count'),
    monto_parado=('MONTO_ACTUALIZADO _PIP', 'sum'),
    poblacion_afectada=('POBLACION_BENEFICIADA_SSP', 'sum')
).sort_values(by='monto_parado', ascending=False).reset_index()

print("=== 3. TOP 10 UNIDADES EJECUTORAS CON MAYOR MONTO EN PARALIZACIÓN ===")
print(ue_paralizadas.head(10).to_string(index=False))

=== 1. RESUMEN DE PROYECTOS EN ENTRAMPAMIENTO (PARALIZADAS Y RESUELTAS) ===
Número de proyectos estancados: 110
Monto total atrapado: S/ 5,258,480,990.54
Población total sin servicio: 3,134,290 habitantes

=== 2. COSTO PROMEDIO DE INVERSIÓN POR PERSONA BENEFICIADA ===
           ESTADO  costo_por_beneficiario
          Cerrado             8859.020324
   En elaboración             5352.953658
Contrato Resuelto             1855.508110
     En Ejecución             1735.676483
       Paralizada             1681.141797
Convenio Resuelto             1624.410840
    Actos Previos             1230.426425
        Concluido              708.086494


=== 3. TOP 10 UNIDADES EJECUTORAS CON MAYOR MONTO EN PARALIZACIÓN ===
                                                 UNIDAD_EJECUTORA DEPARTAMENTO  proyectos_parados  monto_parado  poblacion_afectada
                                       GOBIERNO REGIONAL DE PASCO        PASCO                  1  696746124.40             70334.0
                 